# KDD Process Volcano Data Analysis

In [1]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## Data Cleaning and Preprocessing

In [2]:
def load_data(filepath="volcano-events.tsv"):
    try:
        df = pd.read_csv(filepath, sep='\t')
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return pd.DataFrame()

    df.rename(columns={
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    }, inplace=True)

    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df.dropna(subset=['Year'], inplace=True)
    df['VEI'] = pd.to_numeric(df['VEI'], errors='coerce')
    df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)
    df['Damage_Millions'] = pd.to_numeric(df['Damage_Millions'], errors='coerce').fillna(0)
    df.dropna(subset=['Latitude', 'Longitude', 'Country'], inplace=True)
    return df

## Spatial Analysis

In [3]:
def get_map_figure(df):
    df_map = df.copy()
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    fig = px.scatter_geo(
        df_map,
        lat="Latitude",
        lon="Longitude",
        color="Type",
        size="VEI_Size",
        hover_name="Name",
        hover_data={"Country": True, "Year": True, "Deaths": True, "VEI_Size": False},
        title="Global Volcano Distribution",
        projection="natural earth",
        size_max=15,
        template="plotly_dark",
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_layout(
        margin={"r":0,"t":50,"l":0,"b":0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Temporal Analysis

In [4]:
def get_frequency_figure(df):
    fig = px.histogram(
        df, 
        x="Year", 
        title="Eruption Frequency",
        nbins=100,
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Year", 
        yaxis_title="Count",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Impact Analysis

In [5]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [6]:
def get_correlation_figure(df):
    damage_df = df[df['Damage_Millions'] > 0].copy()
    if damage_df.empty:
         return px.scatter(title="No Data")

    fig = px.scatter(
        damage_df,
        x="VEI",
        y="Damage_Millions",
        size="Deaths",
        hover_name="Name",
        log_y=True,
        title="VEI vs. Impact",
        template="plotly_dark"
    )
    fig.update_traces(marker=dict(color='#ff5722', opacity=0.7))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## UI for Dashboard

In [7]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_data()

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'} # Dropdown text needs to be black to be visible on white bg of default dropdown
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
        html.Div([
            html.Div([dcc.Graph(id='map-graph')], style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='time-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='corr-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'})

    ], style=CONTENT_STYLE)
])

# Callbacks
@app.callback(
    [Output('map-graph', 'figure'),
     Output('time-graph', 'figure'),
     Output('impact-graph', 'figure'),
     Output('corr-graph', 'figure'),
     Output('kpi-eruptions', 'children'),
     Output('kpi-deaths', 'children'),
     Output('kpi-damage', 'children')],
    [Input('country-dropdown', 'value'),
     Input('year-slider', 'value')]
)
def update_dashboard(selected_country, year_range):
    # Filter Data
    dff = df.copy()
    if selected_country:
        dff = dff[dff['Country'] == selected_country]
    
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    if dff.empty:
        dff = df # Fallback if empty

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"

    # Figures
    fig1 = get_map_figure(dff)
    fig2 = get_frequency_figure(dff)
    fig3 = get_impact_figure(dff)
    fig4 = get_correlation_figure(dff)

    return fig1, fig2, fig3, fig4, total_eruptions, total_deaths, total_damage

if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)

Launching Dashboard...
Dashboard launched at: http://127.0.0.1:7860


## Impact Analysis — Top 10 Deadliest Volcanic Eruptions

In this section, we focus on the human impact of volcanic eruptions.  
We analyse which eruptions caused the highest number of fatalities.


In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("volcano-events.tsv", sep="\t")
df.head()


,Search Parameters,Year,Mo,Dy,Tsu,Eq,Name,Location,Country,Latitude,...,Total Deaths,Total Death Description,Total Missing,Total Missing Description,Total Injuries,Total Injuries Description,Total Damage ($Mil),Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
0,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,...,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN


In [3]:
df.columns

Index(['Search Parameters', 'Year', 'Mo', 'Dy', 'Tsu', 'Eq', 'Name',
       'Location', 'Country', 'Latitude', 'Longitude', 'Elevation (m)', 'Type',
       'VEI', 'Agent', 'Deaths', 'Death Description', 'Missing',
       'Missing Description', 'Injuries', 'Injuries Description',
       'Damage ($Mil)', 'Damage Description', 'Houses Destroyed',
       'Houses Destroyed Description', 'Total Deaths',
       'Total Death Description', 'Total Missing', 'Total Missing Description',
       'Total Injuries', 'Total Injuries Description', 'Total Damage ($Mil)',
       'Total Damage Description', 'Total Houses Destroyed',
       'Total Houses Destroyed Description'],
      dtype='object')

In [4]:
import pandas as pd

def build_impact_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocesses the raw volcano events data to construct a dedicated 
    dataset for mortality impact analysis.

    Input:
        df: The original, raw DataFrame containing volcano events.

    Output:
        impact_df: A cleaned DataFrame containing only relevant features:
            - Year
            - Name
            - Country
            - Deaths_clean (standardized total death count)
    """

    # Create a copy to prevent side effects on the original dataframe
    impact_df = df.copy()

    # 1. Schema Resolution: Identify relevant mortality columns.
    # We define a priority list of potential column names to handle schema variations
    # in the input data (e.g., 'Total_Deaths' vs 'DEATHS_TOTAL').
    possible_total_cols = ["Total_Deaths", "TOTAL_DEATHS", "DEATHS_TOTAL"]
    possible_partial_cols = ["Deaths", "DEATHS"]

    # Use a generator to find the first matching column name in the dataframe
    total_col = next((c for c in possible_total_cols if c in impact_df.columns), None)
    partial_col = next((c for c in possible_partial_cols if c in impact_df.columns), None)

    # 2. Feature Engineering: Compute the standardized 'Deaths_clean' metric.
    # We coalesce available columns and handle missing values (NaN) by imputing 0.
    if total_col is not None and partial_col is not None:
        # If both columns exist, sum them to capture the full extent of casualties
        impact_df["Deaths_clean"] = impact_df[total_col].fillna(0) + impact_df[partial_col].fillna(0)
    elif total_col is not None:
        impact_df["Deaths_clean"] = impact_df[total_col].fillna(0)
    elif partial_col is not None:
        impact_df["Deaths_clean"] = impact_df[partial_col].fillna(0)
    else:
        # Critical error handling: Stop execution if no mortality data is found
        raise ValueError("No death-related column found in dataframe.")

    # 3. Feature Selection: Retain only essential variables.
    # We subset the dataframe to keep only the columns required for downstream visualization.
    keep_cols = [c for c in ["Year", "Name", "Country", "Deaths_clean"] if c in impact_df.columns]
    impact_df = impact_df[keep_cols]

    # 4. Data Filtering: Exclude non-fatal events.
    # We filter the dataset to focus exclusively on events with a positive mortality count.
    impact_df = impact_df[impact_df["Deaths_clean"] > 0]

    return impact_df

In [5]:
impact_df = build_impact_dataframe(df)
impact_df.head()


,Year,Name,Country,Deaths_clean
24,-141.0,Etna,Italy,40.0
30,79.0,Vesuvius,Italy,2100.0
36,450.0,Ilopango,El Salvador,30000.0
43,764.0,Aira,Japan,80.0
73,1362.0,Oraefajokull,Iceland,220.0


In [6]:
def build_impact_dataframe(df):
    """
    Constructs a curated dataset focusing on the top 10 deadliest 
    volcanic eruptions for high-level impact analysis.
    """
    # 1. Feature Selection:
    # Subset the dataframe to retain only dimensions relevant to impact analysis 
    # (Metadata + Metrics). Using .copy() avoids SettingWithCopy warnings.
    columns_needed = ['Name', 'Year', 'Country', 'Total Deaths', 'VEI']
    impact_df = df[columns_needed].copy()

    # 2. Data Imputation:
    # Handle missing values (NaN) in the target variable by imputing 0, 
    # assuming NaN implies no recorded deaths.
    impact_df['Total Deaths'] = impact_df['Total Deaths'].fillna(0)

    # 3. Filtering:
    # Apply a boolean mask to keep only lethal events (positive mortality count).
    impact_df = impact_df[impact_df['Total Deaths'] > 0]

    # 4. Ranking:
    # Sort the dataset by the target metric (Total Deaths) in descending order.
    impact_df = impact_df.sort_values(by='Total Deaths', ascending=False)

    # 5. Top-N Sampling:
    # Extract the top 10 records to produce the final leaderboard.
    impact_df = impact_df.head(10)

    return impact_df

In [7]:
import plotly.express as px

def render_impact_chart(impact_df):
    """
    Generates a horizontal bar chart to visualize and rank the deadliest 
    volcanic eruptions.
    """
    
    # 1. Visualization Initialization & Encoding:
    # We use a horizontal bar chart ('h') which is best suited for categorical data 
    # with long labels (Volcano Names), improving readability compared to vertical bars.
    fig = px.bar(
        impact_df,
        x='Total Deaths',      # Quantitative variable (Length of bars)
        y='Name',              # Categorical variable (Y-axis labels)
        orientation='h',       # Horizontal layout
        
        # 2. Visual Weighting:
        # Map the 'Total Deaths' metric to a continuous color scale.
        # This provides a double visual encoding (Length + Color) to emphasize severity.
        color='Total Deaths',
        
        title='Top 10 Deadliest Volcanic Eruptions',
        labels={'Total Deaths': 'Deaths', 'Name': 'Volcano'}
    )

    # 3. Layout Optimization:
    # Adjust the Y-axis sorting logic. 'total ascending' sorts the bars based on their value
    # rather than alphabetically, ensuring an organized leaderboard structure.
    fig.update_layout(yaxis={'categoryorder':'total ascending'})

    return fig

# Execution pipeline
impact_df = build_impact_dataframe(df)
fig = render_impact_chart(impact_df)
fig.show()

### Interpretation – Human Impact of Volcanic Eruptions

The bar chart above displays the Top 10 deadliest volcanic eruptions recorded in the NCEI dataset, based on the *Total Deaths* column.  
The results show that:

- **Tambora (1815)** is by far the deadliest eruption, with nearly **60,000 deaths**, largely due to massive ash fallout and global climate impacts.
- **Krakatau (1883)** follows with over **36,000 deaths**, mostly caused by powerful tsunamis triggered by the explosion.
- **Ilopango**, **Pelee**, and **Nevado del Ruiz** also stand out, each causing more than **20,000 deaths**.
- For some volcanoes such as **Etna** or **Unzendake**, the number of deaths remains high despite lower explosivity indexes (VEI), suggesting that **population density and vulnerability** play a critical role.

This indicator highlights that the human impact of volcanic eruptions depends not only on the physical power of the event, but also on contextual factors such as geography, preparedness, and population exposure.


In [8]:
def render_top10_countries_mortality(df):
    """
    Aggregates and visualizes the total death toll by country, 
    highlighting the 10 most severely affected nations.
    """
    
    # 1. Spatial Aggregation:
    # Group data by political geography ('Country') and compute the cumulative sum 
    # of fatalities. We use .reset_index() to convert the resulting Series back 
    # into a DataFrame for easier plotting.
    country_deaths = df.groupby("Country")["Total Deaths"].sum().reset_index()

    # 2. Ranking & Subsetting:
    # Sort the aggregated data in descending order of magnitude and extract 
    # the top 10 records to focus on the highest-impact regions.
    top10_countries = country_deaths.sort_values(by="Total Deaths", ascending=False).head(10)

    # 3. Visualization:
    # Render a bar chart to compare the cumulative impact across countries.
    fig = px.bar(
        top10_countries,
        x='Country',
        y='Total Deaths',
        title='Top 10 Countries by Total Volcanic Deaths',
        labels={'Total Deaths': 'Cumulative Deaths', 'Country': 'Country'},
        text='Total Deaths'
    )
    
    # Visual Polish: Ensure text labels are readable
    fig.update_traces(textposition='outside')
    
    # Apply log scale if the disparity between the #1 country and #10 is too large
    fig.update_layout(yaxis_type='log')

    return fig

fig = render_top10_countries_mortality(df)
fig.show()

### Countries Most Impacted by Volcanic Fatalities

This chart presents the Top 10 countries that have suffered the highest cumulative number of deaths from volcanic eruptions, based on the *Total Deaths* recorded in the NCEI dataset.

A first observation is the **overwhelming dominance of Indonesia**, which stands far above all other countries with more than **150,000 fatalities**.  
This result is not surprising: Indonesia lies at the intersection of several tectonic plates and hosts over 120 active volcanoes, including some of the most explosive in the world (e.g., Tambora, Krakatau, Kelud, Merapi).

The next countries in the ranking — **El Salvador, Martinique, Colombia, Italy, and Japan** — show much lower totals, yet each of them has experienced at least one historically devastating event:
- **El Salvador**: Ilopango's eruption is among the deadliest in human history.
- **Martinique**: The eruption of Mount Pelée in 1902 destroyed the town of Saint-Pierre, killing over 28,000 people.
- **Colombia**: Nevado del Ruiz (1985) caused nearly 23,000 deaths after lahars buried the town of Armero.
- **Italy and Japan**: Both countries combine high volcanic activity with densely populated regions.

Further down the list, **Guatemala, Iceland, the Philippines, and the United States** also appear due to isolated but impactful eruptions such as Santa María (Guatemala), Laki (Iceland), Pinatubo (Philippines), and St. Helens (USA).

Overall, this analysis shows that:
- **Volcanic fatalities are highly concentrated in a small number of countries**, often those located along active subduction zones.
- **Historical extreme events play a major role** in shaping the long-term fatality totals for each country.
- **Vulnerability factors**, including population density, preparedness, and early warning systems, strongly influence the human impact of eruptions.

This country-level perspective complements the earlier event-level charts by revealing **which regions carry the greatest historical human risk** from volcanic activity.


Scatter plot deaths vs damage

In [9]:
import plotly.express as px

def build_combined_impact_df(df):
    """
    Prepares a consolidated dataset to analyze the intersection of 
    human casualties and economic impact.
    """
    
    # 1. Feature Subsetting:
    # Select specific dimensions for the correlation analysis:
    # - Metadata: Name, Year, Country
    # - Metrics: Total Deaths, Economic Damage
    # - Stratification variable: VEI (Volcanic Explosivity Index)
    combined = df[['Name', 'Year', 'Country', 'Total Deaths', 'Total Damage ($Mil)', 'VEI']].copy()

    # 2. Data Imputation:
    # Fill missing numerical values (NaN) with 0. This assumes that missing data 
    # implies negligible or unrecorded impact, allowing for proper plotting.
    combined['Total Deaths'] = combined['Total Deaths'].fillna(0)
    combined['Total Damage ($Mil)'] = combined['Total Damage ($Mil)'].fillna(0)

    return combined

def render_death_vs_damage(df):
    """
    Generates a multivariate scatter plot to investigate the correlation 
    between human loss (Deaths) and economic loss (Damage).
    """
    
    # 1. Multivariate Visualization:
    # - X-Axis: Human Impact (Deaths)
    # - Y-Axis: Economic Impact (Financial loss in Millions)
    # - Color Encoding: Adds a 3rd dimension (VEI) to visualize if higher intensity
    #   eruptions correlate with higher damages or deaths.
    fig = px.scatter(
        df,
        x='Total Deaths',
        y='Total Damage ($Mil)',
        color='VEI', # Stratify data points by eruption intensity
        
        # 2. Interactive Data Exploration:
        # Add context on hover to identify specific outliers or historical events.
        hover_data=['Name', 'Year', 'Country'],
        
        title='Correlation: Human vs. Economic Impact of Volcanic Eruptions',
        labels={
            'Total Deaths': 'Deaths',
            'Total Damage ($Mil)': 'Damage (Million USD)',
            'VEI': 'Volcanic Explosivity Index (VEI)'
        }
    )

    # 3. Layout Configuration:
    # Set fixed height for readability. 
    # Note: Logarithmic scales (log_x, log_y) are often recommended here due to 
    # extreme outliers, but linear scale is kept as per default.
    fig.update_layout(height=600)

    return fig

# Execution pipeline
combined_df = build_combined_impact_df(df)
fig = render_death_vs_damage(combined_df)
fig.show()

### Distribution of Human Impact


In [10]:
import plotly.express as px

def render_death_boxplot(df):
    """
    Generates a box-and-whisker plot to analyze the statistical distribution 
    and spread of volcanic fatalities.
    """
    
    # 1. Data Preprocessing / Filtering:
    # Isolate events with confirmed fatalities (> 0).
    # This is critical for two reasons:
    #   a) Log(0) is undefined.
    #   b) We are specifically analyzing the severity distribution of *lethal* events.
    df_nonzero = df[df['Total Deaths'] > 0]

    # 2. Statistical Visualization:
    # Initialize a boxplot to visualize central tendency (median) and variability (IQR).
    fig = px.box(
        df_nonzero,
        y='Total Deaths',
        
        # Display all underlying data points (jitter) alongside the box statistics.
        # This reveals the actual data density and prevents the "hiding" of 
        # the specific distribution shape behind the summary statistics.
        points='all',
        
        title='Distribution of Fatalities (Boxplot – Log Scale)',
        labels={'Total Deaths': 'Number of Deaths'},
        
        # 3. Scale Transformation:
        # Apply a logarithmic scale (Log10) to the Y-axis.
        # Mortality data typically follows a "heavy-tailed" or power-law distribution 
        # (many small events, very few massive ones). A linear scale would compress 
        # the box into an unreadable line at the bottom.
        log_y=True
    )

    fig.update_layout(height=600)
    return fig

fig = render_death_boxplot(df)
fig.show()

### Interpretation — Distribution of Fatalities (Log-Scale Boxplot)

This boxplot illustrates the distribution of fatalities caused by volcanic eruptions, using a logarithmic scale to account for the extreme variability in the data.

A first important observation is the **very strong asymmetry** in the distribution.  
The vast majority of eruptions caused **between 1 and 10 deaths**, as shown by the dense cluster of points near the lower end of the scale.  
The median is around **6 deaths**, and the interquartile range (IQR) — which represents the central 50% of events — remains extremely low.

However, the right tail of the distribution reveals the presence of **rare but catastrophic events**, with fatalities in the tens of thousands.  
Examples include:
- **Tambora (1815)** with around **60,000 deaths**,  
- **Krakatau (1883)** with over **36,000 deaths**,  
- **Ilopango**, **Pelee**, and others exceeding several thousand deaths.

These eruptions appear as extreme outliers, far detached from the rest of the distribution.

The log-scale allows these extreme events to be visible while preserving the structure of the lower end of the distribution.  
Without this scaling, the boxplot would be visually dominated by the catastrophic events, making the majority of the data unreadable.

Overall, this analysis highlights that:
- **Volcanic fatality data follow a heavy-tailed distribution**: most eruptions are relatively mild in terms of human loss, but a small number of exceptional events account for the immense majority of fatalities.
- These extreme events play a disproportionate role in shaping global volcanic risk.
- Understanding this distribution is essential, as it shows why historical disasters still dominate long-term human impact assessments, even though modern eruptions tend to be less deadly thanks to improved monitoring and evacuation strategies.

This distribution analysis complements the Top 10 deadliest events by contextualizing how unusual and impactful these catastrophic eruptions truly are.


Répartition des morts PAR TYPE D’ÉRUPTION

In [11]:
import plotly.express as px

def render_death_boxplot_by_type(df):
    """
    Generates a comparative boxplot analysis to examine mortality distributions 
    across different geological classifications of volcanoes.
    """
    
    # 1. Data Cleaning & Cohort Selection:
    # Filter for lethal events only (>0 deaths) to focus on severity analysis.
    df_nonzero = df[df['Total Deaths'] > 0]
    
    # Remove records with missing classification data ('Type') to ensure 
    # the integrity of the categorical grouping.
    df_nonzero = df_nonzero[df_nonzero['Type'].notna()]

    # 2. Comparative Visualization:
    # Initialize a boxplot to contrast statistical distributions (median, IQR)
    # between different volcano types (e.g., Stratovolcano, Shield, etc.).
    fig = px.box(
        df_nonzero,
        x='Type',           # Categorical independent variable
        y='Total Deaths',   # Numerical dependent variable
        
        # Display underlying data points to reveal density and cluster patterns
        # within each category.
        points='all',       
        
        title='Fatalities Distribution by Volcano Type (Log Scale)',
        labels={'Total Deaths': 'Number of deaths', 'Type': 'Volcano Type'},
        
        # 3. Scaling:
        # Apply logarithmic scale to handle the extreme right-skewness of casualty data
        # (orders of magnitude difference between events).
        log_y=True
    )

    fig.update_layout(height=700)

    # 4. Readability Optimization:
    # Rotate x-axis labels by 45 degrees to prevent text overlapping, 
    # which is common with long categorical names.
    fig.update_xaxes(tickangle=45)

    return fig

fig = render_death_boxplot_by_type(df)
fig.show()

mort par an

In [12]:
import pandas as pd
import plotly.express as px

def build_deaths_by_year(df):
    """
    Aggregates global mortality data by year to construct a time-series dataset
    of volcanic impact.
    """
    
    # 1. Feature Selection:
    # Isolate the temporal variable ('Year') and the target metric ('Total Deaths').
    # Using .copy() prevents 'SettingWithCopy' warnings and protects the original dataframe.
    temp = df[['Year', 'Total Deaths']].copy()

    # 2. Data Imputation:
    # Handle missing mortality values (NaN) by assuming zero deaths for those records.
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)

    # 3. Data Integrity Check:
    # Remove records where the timestamp ('Year') is missing, as these cannot be 
    # placed on a chronological timeline. 
    # Note: Valid negative values (BC years) are preserved.
    temp = temp[temp['Year'].notna()]

    # 4. Temporal Aggregation:
    # Group individual volcanic events by year and sum the fatalities to derive
    # the total annual death toll.
    deaths_by_year = temp.groupby('Year')['Total Deaths'].sum().reset_index()

    # 5. Sparsity Reduction / Filtering:
    # Exclude years with zero recorded deaths to focus the analysis strictly
    # on years with confirmed volcanic fatalities.
    deaths_by_year = deaths_by_year[deaths_by_year['Total Deaths'] > 0]

    return deaths_by_year

# --- Execution Pipeline ---

# Generate the aggregated dataset
deaths_year_df = build_deaths_by_year(df)

# Render the visualization (using the function defined previously)
fig = render_deadliest_years_chronological(deaths_year_df)
fig.show()

NameError: name 'render_deadliest_years_chronological' is not defined

In [ ]:
import plotly.express as px

def render_deaths_over_time(df):
    """
    Generates a time-series line chart to visualize the temporal trend 
    and volatility of volcanic fatalities throughout history.
    """
    
    # 1. Time-Series Visualization:
    # Initialize a line plot to observe the longitudinal progression of fatalities.
    # - X-Axis: Time dimension (Year)
    # - Y-Axis: Impact metric (Total Deaths)
    fig = px.line(
        df,
        x='Year',
        y='Total Deaths',
        title='Total Fatalities per Year (Impact Evolution)',
        
        # 2. Data Point Visibility:
        # Enable markers to explicitly indicate recorded data points. 
        # This is crucial for 'sparse' datasets where events do not occur 
        # at regular intervals, preventing misinterpretation of the interpolated lines.
        markers=True,
        
        labels={
            'Year': 'Year',
            'Total Deaths': 'Recorded Deaths'
        }
    )

    # 3. Layout Configuration:
    fig.update_layout(height=500)

    return fig

fig = render_deaths_over_time(deaths_year_df)
fig.show()

In [15]:
import plotly.express as px

def build_deaths_by_region(df):
    """
    Aggregates mortality data by Country and Location to prepare a 
    hierarchical dataset.
    """
    # 1. Feature Selection & Preprocessing:
    # Select the categorical hierarchy (Country -> Location) and the target metric.
    # We use .copy() to ensure data integrity during transformation.
    temp = df[['Country', 'Location', 'Total Deaths']].copy()
    
    # Impute missing values (NaN) with 0 to allow for valid summation.
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)

    # 2. Hierarchical Aggregation:
    # Group data by parent category ('Country') and child category ('Location').
    # This preserves the geographical taxonomy needed for the Sunburst chart.
    grouped = (
        temp.groupby(['Country', 'Location'], as_index=False)['Total Deaths']
            .sum()
    )

    # 3. Filtering & Ranking:
    # Remove zero-impact regions and sort by severity to highlight critical areas.
    grouped = grouped[grouped['Total Deaths'] > 0]
    grouped = grouped.sort_values(by='Total Deaths', ascending=False)

    return grouped

def render_region_sunburst(df_region):
    """
    Generates a Hierarchical Sunburst chart (Country -> Region) to visualize
    the distribution of fatalities across different geographic levels.
    """
    
    # 1. Subsetting:
    # Extract the top 20 most impacted locations. This prevents visual clutter
    # and focuses the analysis on major historical hotspots.
    top_regions = df_region.head(20).copy()

    # 2. Visualization Initialization:
    # Initialize the Sunburst chart with a defined hierarchy.
    fig = px.sunburst(
        top_regions,
        
        # Define the hierarchical path: The inner ring represents 'Country', 
        # and the outer ring represents specific 'Location'.
        path=['Country', 'Location'],  
        
        # Sector size is proportional to the death toll.
        values='Total Deaths',
        
        title='Regions with the Highest Volcanic Fatalities (By Country)',
        
        # 3. Visual Encoding (Color):
        # Map the 'Total Deaths' metric to a color scale for a secondary visual cue.
        color='Total Deaths',
        
        # Use a diverging color scale ('RdBu_r': Red-Blue reversed).
        # Red indicates high mortality (danger), Blue indicates lower mortality relative to the set.
        color_continuous_scale='RdBu_r' 
    )
    
    # 4. Layout Optimization:
    # Increase height to accommodate labels within the concentric rings.
    fig.update_layout(height=700)
    
    return fig

# --- Execution Pipeline ---
df_region = build_deaths_by_region(df)
fig = render_region_sunburst(df_region)
fig.show()

In [17]:
import plotly.express as px

def build_deaths_injuries_df(df):
    """
    Constructs a subset dataframe containing only events with recorded 
    human casualties (deaths or injuries) for correlation analysis.
    """
    # 1. Feature Selection:
    # Extract relevant metrics and metadata. 
    # Included 'Death Description' to enhance hover context in visualizations.
    temp = df[['Name', 'Year', 'Country', 'Total Deaths', 'Total Injuries', 'Death Description']].copy()

    # 2. Data Imputation:
    # Fill NaN values with 0 to allow for numerical comparison.
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)
    temp['Total Injuries'] = temp['Total Injuries'].fillna(0)

    # 3. Filtering:
    # logical OR operator (|) to keep events where at least one form 
    # of casualty occurred.
    temp = temp[(temp['Total Deaths'] > 0) | (temp['Total Injuries'] > 0)]

    return temp

def render_deaths_vs_injuries_bubble(df):
    """
    Generates a multivariate Bubble Chart to visualize the relationship 
    between deaths and injuries, using marker size as a third dimension.
    """
    fig = px.scatter(
        df,
        x='Total Deaths',
        y='Total Injuries',
        
        # 1. Visual Encoding (Size):
        # Map the 'Total Deaths' magnitude to the physical size of the bubble.
        # This emphasizes the most lethal events visually.
        size='Total Deaths',
        
        # 2. Visual Encoding (Color):
        # Map injuries to color to distinguish events with high injuries 
        # but potentially fewer deaths.
        color='Total Injuries',
        
        hover_data=['Name', 'Year', 'Country'],
        title='Bubble Chart: Deaths vs Injuries (Bubble Size = Deaths)',
        labels={
            'Total Deaths': 'Deaths',
            'Total Injuries': 'Injuries'
        }
    )
    fig.update_layout(height=600)
    return fig

# --- Execution ---
death_injury_df = build_deaths_injuries_df(df)
fig1 = render_deaths_vs_injuries_bubble(death_injury_df)
fig1.show()

In [18]:
def render_deaths_vs_injuries_log(df):
    """
    Generates a Log-Log Scatter Plot to analyze the correlation between 
    fatalities and injuries across varying orders of magnitude.
    """
    fig = px.scatter(
        df,
        x='Total Deaths',
        y='Total Injuries',
        
        # Add 'Death Description' to provides context on the cause 
        # (e.g., "Tephra", "Tsunami") when hovering over a point.
        hover_data=['Name', 'Year', 'Country', 'Death Description'], 
        
        title='Deaths vs Injuries Correlation (Log-Log Scale)',
        labels={
            'Total Deaths': 'Deaths',
            'Total Injuries': 'Injuries'
        }
    )

    # 1. Scale Transformation:
    # Apply logarithmic scales to both axes. 
    # This is essential for volcanic data which follows a power-law distribution 
    # (many small events, few massive ones), preventing data bunching near the origin.
    fig.update_layout(
        height=600,
        xaxis_type='log',   
        yaxis_type='log'    
    )

    return fig

# --- Execution ---
# We reuse the 'death_injury_df' created in the previous cell
fig2 = render_deaths_vs_injuries_log(death_injury_df)
fig2.show()

In [20]:
import plotly.express as px

def build_correlation_df(df):
    """
    Prepares a subset of quantitative variables to analyze the statistical 
    correlationships between eruption intensity and impact metrics.
    """
    
    # 1. Feature Selection:
    # Define a list of numerical variables relevant for impact assessment.
    # We include 'VEI' (Intensity) to check if it correlates with casualties or damage.
    cols = [
        'Total Deaths',
        'Total Injuries',
        'Total Damage ($Mil)',
        'Total Houses Destroyed',
        'VEI'
    ]
    
    # 2. Schema Validation:
    # Dynamically filter the list to ensure all selected columns exist in the 
    # current dataframe, preventing KeyErrors if the schema changes.
    cols = [c for c in cols if c in df.columns]
    
    # 3. Data Imputation & Subsetting:
    # - Extract the selected columns.
    # - Fill missing values (NaN) with 0. 
    #   Note: This is necessary because the .corr() method excludes NaN pairs, 
    #   which could lead to an empty or biased matrix in sparse datasets.
    corr_df = df[cols].copy().fillna(0)
    
    return corr_df

def render_correlation_heatmap(df):
    """
    Generates a Heatmap visualization of the Pearson correlation matrix.
    """
    
    # 1. Statistical Computation:
    # Calculate the pairwise correlation coefficients for all columns.
    # Default method is 'Pearson' (linear relationship), ranging from -1 to 1.
    corr_matrix = df.corr()

    # 2. Visualization Initialization:
    fig = px.imshow(
        corr_matrix,
        
        # Display the actual correlation values inside the cells for precision.
        text_auto=True,
        
        # Use a diverging color scale (Red-Blue reversed).
        # - Red (1.0): Strong positive correlation (variables move together).
        # - Blue (-1.0): Strong negative correlation.
        # - White (0.0): No linear correlation.
        color_continuous_scale='RdBu_r',
        
        title='Correlation Matrix – Impact Variables'
    )
    
    fig.update_layout(height=600)
    return fig

# --- Execution Pipeline ---
corr_df = build_correlation_df(df)
fig = render_correlation_heatmap(corr_df)
fig.show()

In [21]:
import plotly.express as px

def build_injury_ratio_df(df):
    """
    Computes a derived 'Injury-to-Death Ratio' to analyze the nature of 
    casualties (e.g., events with high injury counts but low mortality).
    """
    # 1. Feature Selection:
    # Extract the necessary columns for the calculation and metadata for context.
    temp = df[['Name', 'Year', 'Country', 'Total Deaths', 'Total Injuries', 'Death Description']].copy()

    # 2. Data Imputation:
    # Handle missing values by assuming 0 for unrecorded casualties.
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)
    temp['Total Injuries'] = temp['Total Injuries'].fillna(0)

    # 3. Data Integrity & Filtering:
    # Filter the dataset to exclude events with zero deaths.
    # This prevents 'DivisionByZero' errors and undefined values (Infinity) 
    # during the ratio calculation.
    temp = temp[temp['Total Deaths'] > 0]

    # 4. Feature Engineering:
    # Calculate the ratio: Number of injuries per single death.
    # A high ratio suggests an event with high physical impact but lower lethality.
    temp['Injury/Death Ratio'] = temp['Total Injuries'] / temp['Total Deaths']

    # Final cleanup: Remove any remaining invalid entries
    temp = temp[temp['Injury/Death Ratio'].notna()]

    return temp

def render_top10_injury_ratio(df_ratio):
    """
    Visualizes the top 10 volcanic events with the highest proportion 
    of injuries relative to fatalities.
    """
    
    # 1. Ranking:
    # Sort by the calculated ratio (descending) and extract the top 10 outliers.
    top10 = df_ratio.sort_values(by='Injury/Death Ratio', ascending=False).head(10)

    # 2. Visualization:
    # Use a horizontal bar chart to display the leaderboard.
    fig = px.bar(
        top10,
        x='Injury/Death Ratio',
        y='Name',
        orientation='h',
        title='Top 10 Eruptions with Highest Injury/Death Ratio',
        
        # Display the ratio value on the bar for precision
        text='Injury/Death Ratio',
        
        labels={
            'Injury/Death Ratio': 'Injuries per Death',
            'Name': 'Volcano'
        },
        
        # 3. Contextual Data:
        # Include detailed metadata in the tooltip to help interpret WHY 
        # the ratio is high (e.g., specific year or location).
        hover_data=['Year','Country','Total Deaths','Total Injuries','Death Description']
    )

    # 4. Visual Formatting:
    # Format the text position and ensure text displays up to 2 decimal places (optional formatting hint)
    fig.update_traces(textposition='outside', texttemplate='%{text:.2f}')
    
    # Sort the Y-axis to have the highest ratio at the top
    fig.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        height=600
    )

    return fig

# --- Execution Pipeline ---
injury_ratio_df = build_injury_ratio_df(df)
fig = render_top10_injury_ratio(injury_ratio_df)
fig.show()

In [22]:
import plotly.express as px

def render_injury_ratio_scatter(df_ratio):
    """
    Generates a scatter plot to analyze how the Injury-to-Death ratio changes 
    as a function of the total death toll (Event Severity).
    """
    fig = px.scatter(
        df_ratio,
        x='Total Deaths',           # Independent Variable (Magnitude of the event)
        y='Injury/Death Ratio',     # Dependent Variable (Nature of the impact)
        
        # Add metadata to identify specific outliers or historical anomalies
        hover_data=['Name','Year','Country','Total Injuries','Death Description'],
        
        title='Injury/Death Ratio vs Fatalities (Impact Profile)',
        labels={
            'Total Deaths': 'Deaths',
            'Injury/Death Ratio': 'Injuries per Death'
        },
        
        # 1. Scale Transformation:
        # Apply a logarithmic scale to the X-axis (Total Deaths).
        # This is critical to visualize the relationship across a wide dynamic range,
        # allowing comparison between minor incidents (low deaths) and 
        # cataclysmic events (high deaths) on the same canvas.
        log_x=True
    )

    fig.update_layout(height=600)
    return fig

fig = render_injury_ratio_scatter(injury_ratio_df)
fig.show()

In [27]:
import plotly.express as px

def render_region_sunburst(df):
    """
    Generates a multi-level Sunburst chart to visualize the hierarchy of 
    fatalities: Region (Parent) -> Volcano (Child).
    """
    # 1. Feature Selection & Imputation:
    # Work on a copy to calculate specific hierarchical grouping
    temp = df[['Location', 'Name', 'Total Deaths']].copy()
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)

    # 2. Multi-level Aggregation:
    # Group data by both 'Location' AND 'Name' to prepare the hierarchical structure.
    grouped = (
        temp.groupby(['Location', 'Name'], as_index=False)['Total Deaths']
        .sum()
    )

    # 3. Data Sampling (Noise Reduction):
    # Select only the top 50 deadliest volcano-region pairs to avoid visual clutter.
    top = grouped.sort_values(by='Total Deaths', ascending=False).head(50)

    # 4. Hierarchical Visualization:
    fig = px.sunburst(
        top,
        path=['Location', 'Name'],
        values='Total Deaths',
        title='Regional Breakdown of Volcanic Fatalities (Sunburst)'
    )

    fig.update_layout(height=700)
    return fig

# On appelle direct la fonction avec le df brut
fig = render_region_sunburst(df)
fig.show()

In [28]:
import plotly.express as px

def render_deadliest_volcanoes_treemap(df_volcano, top_n=20):
    """
    Generates a Treemap visualization to compare the cumulative lethality 
    of specific volcanoes.
    """
    
    # 1. Data Subsetting:
    # Extract the top N records (default 20) to focus the visualization 
    # on the most critical entities. 
    # Note: Assumes 'df_volcano' is already sorted by 'Total Deaths' (descending).
    top = df_volcano.head(top_n).copy()

    # 2. Visualization Initialization:
    # Initialize a Treemap where each rectangle represents a volcano.
    fig = px.treemap(
        top,
        path=['Name'],          # Categorical variable (Label)
        values='Total Deaths',  # Quantitative variable determining rectangle size (Area)
        
        title=f'Treemap – Top {top_n} Deadliest Volcanoes (All Eruptions Combined)',
        
        # 3. Visual Encoding (Color):
        # Map the 'Total Deaths' metric to color intensity as well as size.
        # This "double encoding" (Size + Color) reinforces the visual hierarchy,
        # making the most lethal volcanoes immediately distinguishable.
        color='Total Deaths',
        
        # Use a sequential color scale ('Reds').
        # Darker shades indicate higher death tolls, intuitively signaling danger.
        color_continuous_scale='Reds'
    )

    # 4. Layout Optimization:
    # Increase height to ensure labels fit within the rectangles.
    fig.update_layout(height=750)

    return fig

# --- Execution ---
fig = render_deadliest_volcanoes_treemap(deaths_by_volcano_df, top_n=20)
fig.show()

In [29]:
import plotly.express as px

def build_deaths_by_volcano(df):
    """
    Aggregates fatalities at the specific volcano level (higher granularity).
    Accumulates deaths across all recorded eruptions for each volcano.
    """
    # 1. Feature Selection:
    # Select the entity identifier ('Name') and the target metric ('Total Deaths').
    temp = df[['Name', 'Total Deaths']].copy()
    
    # 2. Data Imputation:
    # Fill missing values with 0 to ensure accurate summation.
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)

    # 3. Granular Aggregation:
    # Group by volcano 'Name' to calculate the cumulative death toll per volcano.
    # Unlike regional grouping, this identifies specific high-risk geological structures.
    grouped = (
        temp.groupby('Name', as_index=False)['Total Deaths']
            .sum()
    )

    # 4. Filtering & Ranking:
    # Remove volcanoes with zero recorded fatalities and sort by impact magnitude.
    grouped = grouped[grouped['Total Deaths'] > 0]
    grouped = grouped.sort_values(by='Total Deaths', ascending=False)

    return grouped

def render_deadliest_volcanoes_donut(df_volcano, top_n=10):
    """
    Generates a Donut Chart to visualize the proportional distribution 
    (share) of fatalities among the most lethal volcanoes.
    """
    
    # 1. Data Subsetting:
    # Slice the Top N records. This is crucial for Pie/Donut charts, 
    # as displaying too many slices makes the chart unreadable and cognitively overwhelming.
    top = df_volcano.head(top_n).copy()

    # 2. Visualization Initialization:
    # Initialize a Pie chart with a 'hole' property to create a Donut style.
    fig = px.pie(
        top,
        names='Name',           # Categorical labels (Slices)
        values='Total Deaths',  # Quantitative value (Arc size)
        
        # 3. Aesthetic Configuration:
        # The 'hole' argument (0.45) transforms the Pie into a Donut chart.
        # This reduces the visual weight of the center, focusing attention on the arc lengths.
        hole=0.45,
        
        title=f'Share of Total Fatalities Among Top {top_n} Deadliest Volcanoes'
    )

    fig.update_layout(height=600)
    return fig

# --- Execution Pipeline ---
deaths_by_volcano_df = build_deaths_by_volcano(df)

# Check the data structure (Analysis step)
# deaths_by_volcano_df.head() 

fig = render_deadliest_volcanoes_donut(deaths_by_volcano_df, top_n=10)
fig.show()

In [30]:
import plotly.graph_objects as go

def build_pareto_df(df):
    """
    Constructs a dataset specifically engineered for Pareto Analysis:
    Sorts events by impact magnitude and calculates cumulative distribution metrics.
    """
    # 1. Feature Selection & Cleaning:
    # Isolate relevant columns and handle missing values to ensure calculation stability.
    temp = df[['Name', 'Year', 'Country', 'Total Deaths']].copy()
    temp['Total Deaths'] = temp['Total Deaths'].fillna(0)
    
    # 2. Filtering:
    # Restrict analysis to lethal events (>0 deaths) to focus on the "Vital Few".
    temp = temp[temp['Total Deaths'] > 0]

    # 3. Ranking (Crucial Step):
    # Sort data in descending order. A Pareto chart requires the highest impact 
    # categories to be plotted first (left to right).
    temp = temp.sort_values(by='Total Deaths', ascending=False)

    # 4. Cumulative Metrics Calculation:
    # - Cumulative Sum: The running total of deaths.
    # - Cumulative Percentage: The contribution of the top N events to the global total.
    temp['Cumulative Deaths'] = temp['Total Deaths'].cumsum()
    
    total = temp['Total Deaths'].sum()
    temp['Cumulative %'] = 100 * temp['Cumulative Deaths'] / total

    # 5. Indexing:
    # Create a rank sequence (1, 2, 3...) to serve as the ordinal X-axis.
    temp['Rank'] = range(1, len(temp) + 1)

    return temp

def render_pareto_chart(df):
    """
    Generates a Dual-Axis Pareto Chart using Plotly Graph Objects.
    - Bars (Primary Y-Axis): Absolute number of deaths per event.
    - Line (Secondary Y-Axis): Cumulative percentage of total fatalities.
    """
    
    # Initialize an empty figure object (allows for complex multi-trace layering)
    fig = go.Figure()

    # --- Trace 1: Absolute Magnitude (Bars) ---
    fig.add_trace(go.Bar(
        x=df['Rank'],
        y=df['Total Deaths'],
        name='Deaths per Eruption',
        marker_color='rgba(222,45,38,0.8)' # Distinct red for impact
    ))

    # --- Trace 2: Cumulative Distribution (Line) ---
    fig.add_trace(go.Scatter(
        x=df['Rank'],
        y=df['Cumulative %'],
        name='Cumulative Percentage',
        
        # Assign this trace to the secondary Y-axis ('y2')
        yaxis='y2',
        
        mode='lines+markers',
        line=dict(color='black', width=2)
    ))

    # --- Dual-Axis Layout Configuration ---
    fig.update_layout(
        title='Pareto Chart – Distribution of Volcanic Fatalities',
        
        xaxis=dict(
            title='Eruption Rank (sorted by impact)',
            showgrid=True
        ),
        
        # Primary Y-Axis (Left): Count of Deaths
        yaxis=dict(
            title='Deaths per Eruption'
        ),
        
        # Secondary Y-Axis (Right): Percentage (0-100%)
        yaxis2=dict(
            title='Cumulative Percentage (%)',
            overlaying='y',   # Superimpose on top of the primary axis
            side='right',     # Position on the right side of the chart
            range=[0, 105]    # Fixed range to accommodate the 100% line comfortably
        ),
        
        height=650,
        legend=dict(x=0.01, y=0.99) # Position legend inside the chart area
    )

    return fig

# --- Execution Pipeline ---
pareto_df = build_pareto_df(df)
fig = render_pareto_chart(pareto_df)
fig.show()

The simplified Pareto chart reveals that volcanic fatalities follow a highly heavy-tailed distribution: fewer than 10 eruptions account for nearly 90% of all recorded deaths. This pattern confirms that volcanic risk is dominated by rare catastrophic events rather than frequent minor eruptions.

In [32]:
import plotly.express as px

def render_loglog_heavytail(df):
    """
    Generates a Log-Log Rank-Size plot to empirically test for a 
    Power Law (Zipfian) distribution in volcanic lethality.
    """
    
    # 1. Visualization Strategy:
    # We plot Rank (X) vs. Magnitude (Y) on a scatter plot.
    # - If the data follows a Power Law, this plot will form a straight line 
    #   sloping downwards on a log-log scale.
    fig = px.scatter(
        df,
        x='Rank',           # Independent Variable: The order of magnitude (1st, 2nd, 3rd...)
        y='Total Deaths',   # Dependent Variable: The actual impact value
        title='Log-Log Plot – Heavy-Tailed Distribution of Volcanic Fatalities',
        labels={
            'Rank': 'Rank (log scale)',
            'Total Deaths': 'Deaths (log scale)'
        }
    )

    fig.update_layout(height=600)

    # 2. Scale Transformation (The "Log-Log" part):
    # Applying logarithmic scaling to BOTH axes is the standard technique 
    # to linearize a heavy-tailed distribution. 
    # It reveals the underlying structure of data that spans several orders of magnitude.
    fig.update_xaxes(type='log')
    fig.update_yaxes(type='log')

    return fig

# --- Execution ---
fig = render_loglog_heavytail(pareto_df)
fig.show()